# Distributed Hyperparameter Tuning with Kubeflow Trainer

This example demonstrates how to perform hyperparameter tuning by running multiple training experiments
in parallel using [Kubeflow Trainer](https://www.kubeflow.org/docs/components/trainer/overview/).

## Why Hyperparameter Tuning Matters

Model performance depends heavily on hyperparameters such as learning rate, batch size, and network
architecture choices. Finding the right combination can significantly improve accuracy.

## Approach

Instead of relying on external frameworks like Ray Tune or Optuna, this example uses Kubeflow Trainer's
native ability to submit and manage multiple TrainJobs. Each TrainJob runs a different hyperparameter
configuration, and Kubernetes handles the parallel execution.

This approach is:
- **Simple**: No extra dependencies beyond PyTorch
- **Scalable**: Kubernetes can schedule jobs across available cluster resources
- **Isolated**: Each trial runs in its own pod with independent resources and logs
- **Fault-tolerant**: A failed trial does not affect other trials

We train a configurable CNN on the [CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html) dataset,
inspired by the [PyTorch Hyperparameter Tuning Tutorial](https://pytorch.org/tutorials/beginner/hyperparameter_tuning_tutorial.html).

## Install the Kubeflow SDK

You need to install the Kubeflow SDK to interact with Kubeflow Trainer APIs:

In [ ]:
# !pip install -U kubeflow

## Install the PyTorch Dependencies

You also need to install PyTorch and Torchvision to be able to run the example locally:

In [ ]:
!pip install torch==2.9.1
!pip install torchvision==0.22.1

## Define the Training Function

The training function accepts hyperparameters as arguments and trains a configurable CNN on CIFAR-10
using PyTorch Distributed Data Parallel (DDP).

The CNN architecture has two configurable fully connected layer sizes (`l1` and `l2`),
a configurable learning rate (`lr`), and a configurable batch size (`batch_size`).

At the end of training, the function prints a summary line with the configuration and final
test accuracy. This line is used later to compare results across different configurations.

In [ ]:
def train_with_config(lr: float, batch_size: int, l1: int, l2: int, num_epochs: int = 3):
    import os

    import torch
    import torch.distributed as dist
    import torch.nn.functional as F
    from torch import nn
    from torch.utils.data import DataLoader, DistributedSampler
    from torchvision import datasets, transforms

    # Define a configurable CNN model for CIFAR-10
    class ConfigurableCNN(nn.Module):
        def __init__(self, l1=120, l2=84):
            super(ConfigurableCNN, self).__init__()
            self.conv1 = nn.Conv2d(3, 6, 5)
            self.pool = nn.MaxPool2d(2, 2)
            self.conv2 = nn.Conv2d(6, 16, 5)
            self.fc1 = nn.Linear(16 * 5 * 5, l1)
            self.fc2 = nn.Linear(l1, l2)
            self.fc3 = nn.Linear(l2, 10)

        def forward(self, x):
            x = self.pool(F.relu(self.conv1(x)))
            x = self.pool(F.relu(self.conv2(x)))
            x = x.view(-1, 16 * 5 * 5)
            x = F.relu(self.fc1(x))
            x = F.relu(self.fc2(x))
            x = self.fc3(x)
            return x

    # Use NCCL if a GPU is available, otherwise use Gloo as communication backend.
    device, backend = ("cuda", "nccl") if torch.cuda.is_available() else ("cpu", "gloo")
    print(f"Using Device: {device}, Backend: {backend}")

    # Setup PyTorch distributed.
    local_rank = int(os.getenv("LOCAL_RANK", 0))
    dist.init_process_group(backend=backend)
    print(
        "Distributed Training for WORLD_SIZE: {}, RANK: {}, LOCAL_RANK: {}".format(
            dist.get_world_size(),
            dist.get_rank(),
            local_rank,
        )
    )

    # Create the model and load it into the device.
    device = torch.device(f"{device}:{local_rank}")
    model = nn.parallel.DistributedDataParallel(
        ConfigurableCNN(l1=l1, l2=l2).to(device)
    )
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    criterion = nn.CrossEntropyLoss()

    # Data augmentation and normalization for CIFAR-10
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])

    # Download CIFAR-10 dataset only on local_rank=0 process.
    if local_rank == 0:
        train_dataset = datasets.CIFAR10(
            "./data", train=True, download=True, transform=transform_train
        )
        test_dataset = datasets.CIFAR10(
            "./data", train=False, download=True, transform=transform_test
        )
    dist.barrier()
    train_dataset = datasets.CIFAR10(
        "./data", train=True, download=False, transform=transform_train
    )
    test_dataset = datasets.CIFAR10(
        "./data", train=False, download=False, transform=transform_test
    )

    # Shard the dataset across workers.
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        sampler=DistributedSampler(train_dataset),
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        sampler=DistributedSampler(test_dataset, shuffle=False),
    )

    # Training loop
    dist.barrier()
    for epoch in range(1, num_epochs + 1):
        model.train()
        train_loader.sampler.set_epoch(epoch)
        running_loss = 0.0

        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            if batch_idx % 100 == 99 and dist.get_rank() == 0:
                print(
                    "Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.4f}".format(
                        epoch,
                        batch_idx * len(inputs),
                        len(train_loader.dataset),
                        100.0 * batch_idx / len(train_loader),
                        running_loss / 100,
                    )
                )
                running_loss = 0.0

    # Evaluate on test set
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    # Gather results from all ranks
    correct_tensor = torch.tensor([correct], device=device)
    total_tensor = torch.tensor([total], device=device)
    dist.all_reduce(correct_tensor, op=dist.ReduceOp.SUM)
    dist.all_reduce(total_tensor, op=dist.ReduceOp.SUM)
    accuracy = 100.0 * correct_tensor.item() / total_tensor.item()

    dist.barrier()
    if dist.get_rank() == 0:
        print(
            "CONFIG: lr={}, batch_size={}, l1={}, l2={} => Accuracy: {:.2f}%".format(
                lr, batch_size, l1, l2, accuracy
            )
        )

    # Clean up PyTorch distributed
    dist.destroy_process_group()

## Define the Hyperparameter Search Space

We define a small grid of hyperparameter configurations. Each configuration will be submitted
as a separate TrainJob to the Kubeflow Trainer cluster.

The hyperparameters we tune are:
- `lr`: Learning rate for SGD optimizer
- `batch_size`: Mini-batch size for training
- `l1`: Size of the first fully connected layer
- `l2`: Size of the second fully connected layer

In [ ]:
configs = [
    {"lr": 0.1, "batch_size": 64, "l1": 128, "l2": 64},
    {"lr": 0.01, "batch_size": 64, "l1": 128, "l2": 64},
    {"lr": 0.1, "batch_size": 128, "l1": 256, "l2": 128},
    {"lr": 0.01, "batch_size": 128, "l1": 256, "l2": 128},
    {"lr": 0.001, "batch_size": 64, "l1": 64, "l2": 32},
]

print(f"Total configurations to evaluate: {len(configs)}")
for i, config in enumerate(configs):
    print(f"  Config {i}: {config}")

## Run Locally with a Single Configuration

Before submitting to the cluster, we can verify the training function works by running
a single configuration locally.

In [ ]:
from kubeflow.trainer import CustomTrainer, TrainerClient, LocalProcessBackendConfig

# Initialize local backend
backend_config = LocalProcessBackendConfig(cleanup_venv=True)
local_client = TrainerClient(backend_config=backend_config)

# Find the torch-distributed runtime
for runtime in local_client.list_runtimes():
    if runtime.name == "torch-distributed":
        torch_runtime = runtime
        break

# Run a single config locally to verify it works
test_config = {"lr": 0.01, "batch_size": 64, "l1": 128, "l2": 64, "num_epochs": 1}

job_name = local_client.train(
    trainer=CustomTrainer(
        func=train_with_config,
        func_args=test_config,
        packages_to_install=["torch", "torchvision"],
    ),
    runtime=torch_runtime,
)

for logline in local_client.get_job_logs(job_name, follow=True):
    print(logline, end='')

## Scale with Kubeflow TrainJob

Now we submit all hyperparameter configurations as separate TrainJobs to the Kubeflow Trainer cluster.
Each configuration runs as an independent distributed training job, and Kubernetes handles scheduling
and resource allocation.

In [ ]:
from kubeflow.trainer import CustomTrainer, TrainerClient

client = TrainerClient()

## List the Training Runtimes

You can get the list of available Training Runtimes to start your TrainJob.

In [ ]:
for runtime in client.list_runtimes():
    print(runtime)
    if runtime.name == "torch-distributed":
        torch_runtime = runtime

## Submit All Hyperparameter Configurations

Each configuration is submitted as a separate TrainJob. The jobs run in parallel across the cluster.

In [ ]:
job_ids = []
for i, config in enumerate(configs):
    func_args = {**config, "num_epochs": 3}
    job_id = client.train(
        trainer=CustomTrainer(
            func=train_with_config,
            func_args=func_args,
            num_nodes=1,
            resources_per_node={
                "cpu": 2,
                "memory": "4Gi",
                # Uncomment this to distribute the TrainJob using GPU nodes.
                # "nvidia.com/gpu": 1,
            },
        ),
        runtime=torch_runtime,
    )
    job_ids.append((job_id, config))
    print(f"Submitted job {job_id} with config: {config}")

print(f"\nTotal jobs submitted: {len(job_ids)}")

## Monitor All Jobs

Wait for all submitted TrainJobs to complete. Each job will reach either `Succeeded` or `Failed` status.

In [ ]:
for job_id, config in job_ids:
    result = client.wait_for_job_status(name=job_id, status={"Succeeded", "Failed"})
    print(f"Job {job_id} (lr={config['lr']}, bs={config['batch_size']}, "
          f"l1={config['l1']}, l2={config['l2']}): {result.status}")

## Collect and Compare Results

Read the logs from each completed job and extract the final accuracy summary line.
This lets us compare performance across all hyperparameter configurations.

In [ ]:
print("=" * 70)
print("HYPERPARAMETER TUNING RESULTS")
print("=" * 70)

results = []
for job_id, config in job_ids:
    logs = list(client.get_job_logs(job_id))
    # Find the CONFIG summary line printed at the end of training
    result_lines = [line for line in logs if "CONFIG:" in line]
    if result_lines:
        print(result_lines[-1])
        # Parse accuracy from the summary line
        accuracy_str = result_lines[-1].split("Accuracy: ")[-1].replace("%", "").strip()
        try:
            accuracy = float(accuracy_str)
            results.append({"config": config, "accuracy": accuracy, "job_id": job_id})
        except ValueError:
            print(f"  Could not parse accuracy for job {job_id}")
    else:
        print(f"Job {job_id}: No results found (job may have failed)")

# Identify the best configuration
if results:
    print("\n" + "=" * 70)
    best = max(results, key=lambda x: x["accuracy"])
    print(
        f"BEST CONFIG: lr={best['config']['lr']}, batch_size={best['config']['batch_size']}, "
        f"l1={best['config']['l1']}, l2={best['config']['l2']} "
        f"=> Accuracy: {best['accuracy']:.2f}% (job: {best['job_id']})"
    )
    print("=" * 70)

## Next Steps

This example demonstrated a simple grid search approach to hyperparameter tuning using Kubeflow Trainer.
Here are some ways to extend this pattern:

- **Random search**: Generate configurations programmatically by sampling from distributions
  (e.g., log-uniform for learning rate, choice for batch size)
- **Larger search spaces**: Submit more configurations and let the Kubernetes cluster autoscaler
  provision nodes on demand
- **Katib integration**: For more advanced HPO (Bayesian optimization, early stopping, neural
  architecture search), consider [Katib](https://www.kubeflow.org/docs/components/katib/overview/),
  Kubeflow's dedicated hyperparameter optimization component
- **Optuna**: For Bayesian optimization within a single training job, integrate
  [Optuna](https://optuna.org/) into the training function
- **Multi-node trials**: Each configuration in this example uses a single node, but you can
  increase `num_nodes` for larger models that benefit from distributed data parallelism

## Clean Up

Delete all the TrainJobs created during the hyperparameter search.

In [ ]:
# for job_id, config in job_ids:
#     client.delete_job(job_id)
#     print(f"Deleted job {job_id}")